## In this notebook:

#### We concatenate parcellated PET images into region x receptor matrix of densities.
Adapted from Hansen Receptors (https://github.com/netneurolab/hansen_receptors)
Paper: https://www.nature.com/articles/s41593-022-01186-3

#Fix - output figures.

### Import Packages

In [1]:
import numpy as np
import pandas as pd
from netneurotools import datasets, plotting
from matplotlib.colors import ListedColormap
from scipy.stats import zscore
from nilearn.datasets import fetch_atlas_schaefer_2018
from netneurotools.datasets import fetch_cammoun2012

In [7]:
import surfer

### Set Variables

In [2]:
path = '/Users/pecsok/projects/Neuromaps/pecsok_pfns/neuromaps/'
datapath = '/Users/pecsok/projects/Neuromaps/hansen_receptors/'
#figpath = '/Users/pecsok/Desktop/ImageData/PMACS_remote/data/nmaps/analyses/figures/'
figpath_box = '/Users/pecsok/Library/CloudStorage/Box-Box/GluCEST PhD/Manuscripts/Neuromaps/Results'
atlas = 'cammoun2012' #'schaefer'

if atlas == 'schaefer':
    scale='scale1000_17'
    schaefer = fetch_atlas_schaefer_2018(n_rois=1000, yeo_networks=17) # This is sklearn.utils._bunch.Bunch. Other altases in this format? *Ask Golia
    nnodes = len(schaefer['labels'])
elif atlas == 'cammoun2012':
    scale = 'scale500' #'scale500' #'scale1000_17'  
    folder = 'atl-Cammoun2012_res-500'
    cammoun = fetch_cammoun2012()
    nnodes=1015


In [4]:
print(path+'data/parcellated/PET_parcellated/'+scale+'/mGluR5_abp_hc73_smart.csv')

/Users/pecsok/projects/Neuromaps/pecsok_pfns/neuromaps/data/parcellated/PET_parcellated/scale1000_17/mGluR5_abp_hc73_smart.csv


### Choose what to analyse

In [3]:
receptors_csv = [path+'data/parcellated/PET_parcellated/'+atlas+'_'+scale+'/NMDA_ge179_hc29_galovic.csv',
                 path+'data/parcellated/PET_parcellated/'+atlas+'_'+scale+'/mGluR5_abp_hc22_rosaneto.csv',
                 path+'data/parcellated/PET_parcellated/'+atlas+'_'+scale+'/mGluR5_abp_hc28_dubois.csv',
                 path+'data/parcellated/PET_parcellated/'+atlas+'_'+scale+'/mGluR5_abp_hc73_smart.csv',
                # path+'data/parcellated/PET_parcellated/'+atlas+scale+'/GABAa-bz_flumazenil_hc16_norgaard.csv',
                 path+'data/parcellated/PET_parcellated/'+atlas+'_'+scale+'/GABAa_flumazenil_hc6_dukart.csv',
                 path+'data/parcellated/PET_parcellated/'+atlas+'_'+scale+'/CB1_omar_hc77_normandin.csv'
                ]

### Make Receptor Matrices

In [6]:
print(scale) #pecsok_pfns/neuromaps/data/parcellated/PET_parcellated/"+scale+"

cammoun2012_scale500


In [4]:
# combine all the receptors (including repeats)
r = np.zeros([nnodes, len(receptors_csv)])
print(r.shape)
for i in range(len(receptors_csv)):
    r[:, i] = np.genfromtxt(receptors_csv[i], delimiter=',') 
    # Now we have the receptor data by parcel (row) for all 6 receptor maps (col)

#print(r[:,3])
print(r.shape)

receptor_names = np.array(["NMDA","mGluR5","GABAa","CB1"]) #,"D2"
np.save(path+'data/receptor_names_pet.npy', receptor_names)

# make final region x receptor matrix
receptor_data = np.zeros([nnodes, len(receptor_names)])

# NMDA Data
receptor_data[:, 0] = zscore(r[:, 0])
# weighted average of mGluR5 ABP688
receptor_data[:, 1] = (zscore(r[:, 1])*22 + zscore(r[:, 2])*28 + zscore(r[:, 3])*73) / (22+28+73)
# weighted average of GABA 
receptor_data[:, 2] = zscore(r[:, 4])
# weighted average of CB1
receptor_data[:, 3] = zscore(r[:, 5])

# Make df and add parcel labels
receptor_df = pd.DataFrame(receptor_data, columns=receptor_names)
#schaefer = fetch_atlas_schaefer_2018(n_rois=1000, yeo_networks=17)
#labels = schaefer.labels
#labels = [label.decode('utf-8') for label in labels]

cam = fetch_cammoun2012()
info = pd.read_csv(cammoun['info'], sep=',')
info = info[info['scale']== 'scale500']
hemisphere = info['hemisphere'].values
labels = info['label'].values + hemisphere
print(labels)


receptor_df.index = labels
receptor_df.index.name = 'Parcel'


# Save df
receptor_df.to_csv(path+'results/receptor_data_'+atlas+'_'+scale+'.csv', index=True, header=True)

#print(receptor_df)

(1015, 6)
(1015, 6)
['lateralorbitofrontal_9R' 'lateralorbitofrontal_11R'
 'lateralorbitofrontal_5R' ... 'hippocampusL' 'amygdalaL' 'brainstemL']


In [6]:
"""
plot receptor data
"""

# colourmaps
cmap = np.genfromtxt(datapath+'data/colourmap.csv', delimiter=',')
cmap_div = ListedColormap(cmap)

# Plot each receptor map

## Camoun2012
if atlas == 'cammoun2012':
    cammoun = datasets.fetch_cammoun2012(version='fsaverage')
    cammoun = cammoun['scale500']
    print(cammoun)
    """
    for k in range(len(receptor_names)):
        brain = plotting.plot_fsaverage(data=receptor_data[:, k],
                                        lhannot = cammoun[0],
                                        rhannot = cammoun[1],
                                        colormap='viridis',
                                        views=['lat', 'med'],
                                        data_kws={'representation': "wireframe"})
        #brain.save_image(figpath + folder + '/surface_receptor_'+receptor_names[k]+'.png')
        brain.save_image(figpath_box + '/surface_receptor_'+atlas+'_'+receptor_names[k]+'.png')

    """
    # Loop through receptor names and plot
    for k in range(len(receptor_names)):
        # Create the brain surface plot
        brain = plotting.plot_fsaverage(
            data=receptor_data[:, k],
            lhannot=cammoun[0],
            rhannot=cammoun[1],
            colormap='viridis',
            views=['lat', 'med'],
            data_kws={'representation': "wireframe"}
        )

        # Save the image
        output_file = f"{figpath_box}/surface_receptor_{atlas}_{receptor_names[k]}.png"
        try:
            brain.save_image(output_file)
            print(f"Saved image: {output_file}")
        except Exception as e:
            print(f"Error saving image {output_file}: {e}")

        # Clear the plot to free memory
        brain.close()


## Schaefer1000
if scale == 'scale1000_17':
    annot = datasets.fetch_schaefer2018('fsaverage')['1000Parcels17Networks']
    type(annot)
    print(annot)
    
    for k in range(len(receptor_names)):
        brain = plotting.plot_fsaverage(data=receptor_data[:, k],
                                        lhannot=annot.lh,
                                        rhannot=annot.rh,
                                        colormap='plasma',
                                        views=['lat', 'med'],
                                        data_kws={'representation': "wireframe"})
        brain.save_image(figpath_box + '/surface_receptor_'+atlas+'_'+receptor_names[k]+'.png')



#receptor_data[:, 2:9] = r[:, 3:10]
print(receptor_data)
print(len(receptor_data))

print(r)

Surface(lh='/Users/pecsok/nnt-data/atl-cammoun2012/fsaverage/atl-Cammoun2012_space-fsaverage_res-500_hemi-L_deterministic.annot', rh='/Users/pecsok/nnt-data/atl-cammoun2012/fsaverage/atl-Cammoun2012_space-fsaverage_res-500_hemi-R_deterministic.annot')


qt.qpa.window: <QNSWindow: 0x44d75ab10; contentView=<QNSView: 0x44d75a710; QCocoaWindow(0x600000917de0, window=QWidgetWindow(0x600005b97d80, name="QMainWindowClassWindow"))>> has active key-value observers (KVO)! These will stop working now that the window is recreated, and will result in exceptions when the observers are removed. Break in QCocoaWindow::recreateWindowIfNeeded to debug.
qt.qpa.window: <QNSWindow: 0x39ba287b0; contentView=<QNSView: 0x39ba283b0; QCocoaWindow(0x60000098ee10, window=QWidgetWindow(0x600009a1f420, name="QMainWindowClassWindow"))>> has active key-value observers (KVO)! These will stop working now that the window is recreated, and will result in exceptions when the observers are removed. Break in QCocoaWindow::recreateWindowIfNeeded to debug.
qt.qpa.window: <QNSWindow: 0x376311bf0; contentView=<QNSView: 0x37631db20; QCocoaWindow(0x600000916260, window=QWidgetWindow(0x600005b56580, name="QMainWindowClassWindow"))>> has active key-value observers (KVO)! These wil

Saved image: /Users/pecsok/Library/CloudStorage/Box-Box/GluCEST PhD/Manuscripts/Neuromaps/Results/surface_receptor_cammoun2012_NMDA.png


qt.qpa.window: <QNSWindow: 0x44dfd96d0; contentView=<QNSView: 0x44dfd92d0; QCocoaWindow(0x6000009a1fa0, window=QWidgetWindow(0x600009a60de0, name="QMainWindowClassWindow"))>> has active key-value observers (KVO)! These will stop working now that the window is recreated, and will result in exceptions when the observers are removed. Break in QCocoaWindow::recreateWindowIfNeeded to debug.
qt.qpa.window: <QNSWindow: 0x44c24f9f0; contentView=<QNSView: 0x44c24f5f0; QCocoaWindow(0x600000979080, window=QWidgetWindow(0x600009afe4c0, name="QMainWindowClassWindow"))>> has active key-value observers (KVO)! These will stop working now that the window is recreated, and will result in exceptions when the observers are removed. Break in QCocoaWindow::recreateWindowIfNeeded to debug.
qt.qpa.window: <QNSWindow: 0x37780d890; contentView=<QNSView: 0x37780d490; QCocoaWindow(0x6000009abc80, window=QWidgetWindow(0x600009adcd80, name="QMainWindowClassWindow"))>> has active key-value observers (KVO)! These wil

Saved image: /Users/pecsok/Library/CloudStorage/Box-Box/GluCEST PhD/Manuscripts/Neuromaps/Results/surface_receptor_cammoun2012_mGluR5.png


qt.qpa.window: <QNSWindow: 0x376370780; contentView=<QNSView: 0x376370380; QCocoaWindow(0x600000974160, window=QWidgetWindow(0x600005ac0c00, name="QMainWindowClassWindow"))>> has active key-value observers (KVO)! These will stop working now that the window is recreated, and will result in exceptions when the observers are removed. Break in QCocoaWindow::recreateWindowIfNeeded to debug.
qt.qpa.window: <QNSWindow: 0x45b7c9970; contentView=<QNSView: 0x45b7c9570; QCocoaWindow(0x60000097f2e0, window=QWidgetWindow(0x600005b2a820, name="QMainWindowClassWindow"))>> has active key-value observers (KVO)! These will stop working now that the window is recreated, and will result in exceptions when the observers are removed. Break in QCocoaWindow::recreateWindowIfNeeded to debug.
qt.qpa.window: <QNSWindow: 0x501837780; contentView=<QNSView: 0x501837380; QCocoaWindow(0x600000946680, window=QWidgetWindow(0x600005b7c5a0, name="QMainWindowClassWindow"))>> has active key-value observers (KVO)! These wil

Saved image: /Users/pecsok/Library/CloudStorage/Box-Box/GluCEST PhD/Manuscripts/Neuromaps/Results/surface_receptor_cammoun2012_GABAa.png


qt.qpa.window: <QNSWindow: 0x5aae22fb0; contentView=<QNSView: 0x5aae22bb0; QCocoaWindow(0x6000009a49a0, window=QWidgetWindow(0x60000940ad60, name="QMainWindowClassWindow"))>> has active key-value observers (KVO)! These will stop working now that the window is recreated, and will result in exceptions when the observers are removed. Break in QCocoaWindow::recreateWindowIfNeeded to debug.
qt.qpa.window: <QNSWindow: 0x38d82e420; contentView=<QNSView: 0x38d82e020; QCocoaWindow(0x6000009a8000, window=QWidgetWindow(0x600009477f60, name="QMainWindowClassWindow"))>> has active key-value observers (KVO)! These will stop working now that the window is recreated, and will result in exceptions when the observers are removed. Break in QCocoaWindow::recreateWindowIfNeeded to debug.
qt.qpa.window: <QNSWindow: 0x38d8879a0; contentView=<QNSView: 0x38d8875a0; QCocoaWindow(0x6000009e2470, window=QWidgetWindow(0x60000944bd20, name="QMainWindowClassWindow"))>> has active key-value observers (KVO)! These wil

Saved image: /Users/pecsok/Library/CloudStorage/Box-Box/GluCEST PhD/Manuscripts/Neuromaps/Results/surface_receptor_cammoun2012_CB1.png
[[-0.02086797  0.16462124 -0.15022788  0.53724607]
 [ 0.36601258 -0.0853896   0.17048341  0.81470193]
 [ 1.17184921  1.01544628  1.31715741  0.90826151]
 ...
 [ 0.85539512 -1.00922289 -1.03965146  0.0702233 ]
 [ 0.10952881 -0.78241158 -0.97148672  1.04332287]
 [-0.79893758 -4.70604028 -5.10899002 -3.23990512]]
1015
[[ 6.30872320e+00  3.60468964e+00  2.77503733e+00  7.55093270e-01
   6.18744121e+01  1.35334982e+00]
 [ 6.75982295e+00  3.13090897e+00  2.62848026e+00  7.97383492e-01
   6.48180109e+01  1.38587191e+00]
 [ 7.69942227e+00  3.83834514e+00  2.92945047e+00  9.25286237e-01
   7.53425800e+01  1.39683853e+00]
 ...
 [ 7.33043923e+00  2.85463526e+00  2.34415652e+00  6.47032100e-01
   5.37109760e+01  1.29860757e+00]
 [ 6.46076483e+00  3.14520325e+00  2.41282438e+00  6.48180838e-01
   5.43366155e+01  1.41266980e+00]
 [ 5.40150000e+00  1.94463151e+00  1.4

Context leak detected, msgtracer returned -1
